In [7]:
from typing import Annotated

from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage

In [ ]:
load_dotenv()

In [ ]:
llm = ChatOpenAI(model = 'gpt-4o-mini')

In [8]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages = Annotated[list[BaseMessage], add_messages]


In [10]:
def chat_node(state: ChatState):

    decision = interrupt(
        {
            "type":"approval",
            "reason" : "model is about to answer a user question",
            "question" : state["messages"][-1].content,
            'instructions' : "approve this question ? yes / no"
        }
    )

    if decision["approved"] == "no":
        return {"messages" : [AIMessage(content = "not approved")]}
    else:
        response = llm.invoke(state["messages"])
        return {"messages" : [response]}


In [ ]:
builder = StateGraph(ChatState)

builder.add_node('chat', chat_node)
builder.add_edge(START, 'chat')
builder.add_edge('chat', END)

checkpointer = MemorySaver()

app = builder.compile(checkpointer = checkpointer)

In [ ]:
app

In [ ]:
# Create a new thread id for this conversation
config = {"configurable": {"thread_id": '1234'}}

# ---- STEP 1: user asks a question ----
initial_input = {
    "messages": [
        ("user", "Explain gradient descent in very simple terms.")
    ]
}

# Invoke the graph for the first time
result = app.invoke(initial_input, config=config)

In [ ]:
result

In [ ]:
message = result['__interrupt__'][0].value
message

In [ ]:
user_input = input(f"\nBackend message - {message} \n Approve this question? (y/n): ")

In [ ]:
# Resume the graph with the approval decision
final_result = app.invoke(
    Command(resume={"approved": user_input}),
    config=config,
)

In [ ]:
print(final_result["messages"][-1].content)